# 09｜Valuation PE/PB、財務證據映射 Gate

**執行環境：Google Colab（Python 3）**。會讀取 `janusChatGPT` 研究資料及使用 GitHub `main` 的**固定 SHA**，結果只寫入研究 run／報告目錄。不會修改來源 canonical、Risk ACTIVE、GCP、Registry、PG 或 Core/Mart。

請由上往下執行每一個 cell；首次使用先完成 Google Drive OAuth。

**05.1 關係**：本輪沒有估值新因子，仍使用 Stage 05 Valuation 原 5 個正式席位和原 X/Y SHA；05.1 結果須存在但不能冒充 Valuation 45 欄擴充。財務 mapping 與產業 cohort 證據缺失時保持 BLOCKED。


In [ ]:
# 此設定每支 Notebook 可獨立執行；中途更換 Colab Runtime 也能接續同一研究 run。
import sys, os, json, hashlib, subprocess, traceback
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '沒有找到給ai專用/janusChatGPT；不能改用個人用資料夾'
DATASET = ROOT/'資料源'/'JanusDataset_v3_202610'
MARKET = ROOT/'資料源'/'大盤產業_免費來源_2025H1_試跑'
RUN_HOME = ROOT/'模型研究'/'_colab_pipeline_runs'
ACTIVE = RUN_HOME/'ACTIVE_RUN.json'
REPO = Path('/content/janus-omniforge')
REPO_URL = 'https://github.com/tommylin15/janus-omniforge.git'
RUN_HOME.mkdir(parents=True, exist_ok=True)

def sh(*args, check=True, capture=False):
    return subprocess.run([str(x) for x in args],check=check,text=True,
                          capture_output=capture)

def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str,allow_nan=False)+'\n',encoding='utf-8')
    tmp.replace(path)

def read_json(path):return json.loads(Path(path).read_text(encoding='utf-8'))

def packages():
    # Colab 預裝 pandas/numpy/scikit-learn；只有缺少依賴才安裝。不引入付費服務。
    from importlib.util import find_spec
    missing=[(module,pip) for module,pip in [('pyarrow','pyarrow'),('sklearn','scikit-learn'),('xgboost','xgboost')]
             if find_spec(module) is None]
    if missing:sh(sys.executable,'-m','pip','install','-q',*[p for _,p in missing])
    import pandas, numpy, pyarrow, sklearn, xgboost
    return {k:__import__(k).__version__ for k in ['pandas','numpy','pyarrow','sklearn','xgboost']}

def repo_git(*args):
    # 優先使用公開存取；若 repo 是 private，使用 Colab Secrets 的 GITHUB_TOKEN。
    cmd=['git',*[str(x) for x in args]]
    env=dict(os.environ,GIT_TERMINAL_PROMPT='0')
    result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode!=0:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
        if not token:raise RuntimeError('GIT_ACCESS_FAILED：repo 可能是 private；請設 Colab Secret GITHUB_TOKEN，或檢查存取權')
        import tempfile
        with tempfile.TemporaryDirectory() as folder:
            ask=Path(folder)/'askpass.sh'
            ask.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token;; *) printf %s "$GITHUB_TOKEN";; esac\n')
            ask.chmod(0o700)
            env.update(GIT_ASKPASS=str(ask),GITHUB_TOKEN=token)
            result=subprocess.run(cmd,env=env,text=True,capture_output=True)
    if result.returncode:raise RuntimeError('GIT_COMMAND_FAILED：'+result.stderr[-500:])
    return result

def checkout(code_sha=None):
    if not (REPO/'.git').is_dir():repo_git('clone',REPO_URL,REPO)
    else:repo_git('-C',REPO,'fetch','origin','main')
    if code_sha:
        # 不能用最新 main 偷換固定研究程式版本。
        existing=sh('git','-C',REPO,'cat-file','-e',code_sha+'^{commit}',check=False)
        if existing.returncode:repo_git('-C',REPO,'fetch','origin',code_sha)
        repo_git('-C',REPO,'checkout','--detach','--force',code_sha)
    else:repo_git('-C',REPO,'checkout','--detach','--force','origin/main')
    actual=sh('git','-C',REPO,'rev-parse','HEAD',capture=True).stdout.strip()
    if code_sha and actual!=code_sha:raise RuntimeError('CODE_SHA_CHANGED')
    assert (REPO/'research/chatgpt/label_four_models.py').is_file()
    assert (REPO/'research/chatgpt/janus_train.py').is_file()
    code=str(REPO/'research/chatgpt')
    if code not in sys.path:sys.path.insert(0,code)
    return actual

def open_run():
    if not ACTIVE.is_file():raise RuntimeError('先執行 03_收檔驗收與同步.ipynb，建立研究 run')
    cfg=read_json(ACTIVE)
    out=RUN_HOME/cfg['run_id']
    if not out.is_dir():raise RuntimeError('RUN_FOLDER_MISSING')
    if not (DATASET/'canonical/CURRENT.json').is_file():raise RuntimeError('CURRENT_MISSING')
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED：來源版本已切換，請勿混用研究產物')
    manifest=DATASET/cfg['manifest_rel']
    if sha_file(manifest)!=cfg['manifest_sha256']:
        raise RuntimeError('MANIFEST_CHANGED：固定 release 有變動')
    checkout(cfg['git_sha'])
    return cfg,out

def save_stage(out,n,status,details):
    obj={'stage':n,'status':status,'updated_at_utc':datetime.now(timezone.utc).isoformat(),
         'publication_allowed':False,'details':details}
    write_json(Path(out)/f'{n}_status.json',obj)
    print(json.dumps(obj,ensure_ascii=False,indent=2,default=str)[:12000]);return obj

def stage(out,n):
    p=Path(out)/f'{n}_status.json'
    return read_json(p) if p.is_file() else {'stage':n,'status':'NOT_RUN'}

def ensure_stage(out,n,allow=('PASS',)):
    status=stage(out,n)['status']
    if status not in allow:raise RuntimeError(f'PREVIOUS_GATE_{n}={status}; allowed={allow}')

def keep_current(cfg):
    if sha_file(DATASET/'canonical/CURRENT.json')!=cfg['current_sha256']:
        raise RuntimeError('CURRENT_CHANGED_DURING_STAGE')

def fail_stage(out,n,exc):
    save_stage(out,n,'BLOCKED',{'error_type':type(exc).__name__, 'message':str(exc)[:1200]})
    raise exc

versions=packages()
print('Dependency versions:',versions)


In [ ]:
# 09：valuation 研究訓練。
# 現有 janus_train.run 對 valuation 執行 chronological split、baseline、XGBoost、OOS (retrospective)。
# 這是研究候選，不切 Risk ACTIVE、不自動 promotion。
cfg,out=open_run();ensure_stage(out,'05',('PASS','PARTIAL'))
# Stage 05.1 本輪只擴充 Quant/Risk/Fundamental；Valuation 沒有新增候選。
precheck=read_json(out/'05.1_status.json') if (out/'05.1_status.json').is_file() else None
assert precheck and precheck.get('status')=='PASS_RESEARCH_ONLY', 'STAGE05_1_NOT_PASSED'
assert precheck.get('version')=='05.1.1', 'STAGE05_1_VERSION_MISMATCH'
assert int(precheck.get('candidate_slots',-1))==11, 'STAGE05_1_CANDIDATE_CONTRACT_DRIFT'
import janus_train
import janus_precision
role='valuation'
try:
    slots=read_json(out/'05_feature_slots.json')
    assert slots[role]['status']=='RESEARCH_DERIVED', f'ROLE_XY_NOT_READY: {role}'
    assert slots[role]['x_slots']==5 and slots[role]['features']==janus_precision.VAL, 'VALUATION_REMAINS_5_FEATURES'
    # 只有估值可讀外部提供的 evidence-backed mapping，禁止虛構。
    mapping=None
    if role=='valuation':
        mapping_file=ROOT/'模型研究'/'_config'/'valuation_financial_mapping.json'
        assert mapping_file.is_file(), 'VALUATION_MAPPING_MISSING：請提供帶 evidence 的完整 mapping'
        mapping=read_json(mapping_file)
        assert all(k in mapping.get('metrics',{}) for k in ['roe','debt_ratio','log_total_assets','revenue_yoy'])
        assert mapping.get('cohorts') and mapping.get('calendar_evidence'), 'VALUATION_COHORT_OR_CALENDAR_EVIDENCE_MISSING'
    report,model_dir=janus_train.run(DATASET,out/'training',role,
                                    mode='retrospective',horizon=20,
                                    financial_mapping=mapping,max_trees=400)
    # 若同程式與輸入仍一致，stage05 X/Y 與 train X/Y 應逐位元組相符。
    xy_dir=out/'xy'/role
    for file in ['X.parquet','Y_NOT_X.parquet']:
        original=xy_dir/file;trained=model_dir/'data'/file
        assert original.is_file() and trained.is_file(), f'XY_NOT_FOUND: {file}'
        assert sha_file(original)==sha_file(trained), f'XY_DRIFT: {role}/{file}'
    keep_current(cfg)
    write_json(out/f'09_train.json',{'role':role,'model_dir':str(model_dir),
               'report_state':report.get('state'),'result_file':str(model_dir/'reports/result.json'),
               'git_sha':cfg['git_sha'],'feature_slots':5,
               'source_stage':'05_VALUATION_UNCHANGED','publication_allowed':False})
    eligible=report.get('state')=='RESEARCH_EVALUATED'
    save_stage(out,'09','PASS' if eligible else 'PARTIAL',
               {'role':role,'state':report.get('state'),'model_dir':str(model_dir),
                 'models':{k:v.get('state') for k,v in report.get('models',{}).items()},
                 'retrospective':True,'historical_pit':'NOT_VERIFIED',
                 'feature_slots':5,'source_stage':'05_VALUATION_UNCHANGED'})
except Exception as exc:fail_stage(out,'09',exc)
